# VQE: H2 Ground State (PennyLane)

Variational Quantum Eigensolver for the H2 ground state using a hardware-efficient ansatz with parameter-shift gradients and COBYLA optimisation.

In [ ]:
import numpy as np
import pennylane as qml
import scipy.optimize as opt

## H2 Hamiltonian and ansatz

In [ ]:
N_QUBITS = 2
N_LAYERS = 3
dev = qml.device("default.qubit", wires=N_QUBITS)

H2_HAMILTONIAN = qml.Hamiltonian(
    [-0.81261, 0.17120, -0.22279, 0.17120, 0.04532],
    [qml.Identity(0), qml.Z(0), qml.Z(1),
     qml.Z(0) @ qml.Z(1), qml.X(0) @ qml.X(1)],
)
EXACT_GS_ENERGY = -1.380398

@qml.qnode(dev, diff_method="parameter-shift")
def ansatz(params):
    for layer in range(N_LAYERS):
        base = layer * 4
        qml.RY(params[base + 0], wires=0)
        qml.RZ(params[base + 1], wires=0)
        qml.RY(params[base + 2], wires=1)
        qml.RZ(params[base + 3], wires=1)
        qml.CNOT(wires=[0, 1])
    return qml.expval(H2_HAMILTONIAN)

def energy(params):
    return float(ansatz(params))

## COBYLA optimisation

In [ ]:
rng = np.random.default_rng(42)
init_params = rng.uniform(0, 2 * np.pi, size=4 * N_LAYERS)
print(f"Initial energy: {energy(init_params):.6f}")
print(f"Exact GS energy: {EXACT_GS_ENERGY:.6f}")

history = []
def callback(xk):
    history.append(energy(xk))

result = opt.minimize(
    energy, init_params, method="COBYLA",
    options={"maxiter": 200, "rhobeg": 0.5}, callback=callback,
)

print(f"\nOptimised energy: {result.fun:.6f}")
print(f"Error vs exact:   {abs(result.fun - EXACT_GS_ENERGY):.6f}")

## Energy convergence and final state

In [ ]:
step = max(1, len(history) // 10)
for i in range(0, len(history), step):
    print(f"  iter {i + 1:>3d}  energy = {history[i]:.6f}")
if (len(history) - 1) % step != 0:
    print(f"  iter {len(history):>3d}  energy = {history[-1]:.6f}")

@qml.qnode(dev)
def final_circuit(params):
    for layer in range(N_LAYERS):
        base = layer * 4
        qml.RY(params[base + 0], wires=0)
        qml.RZ(params[base + 1], wires=0)
        qml.RY(params[base + 2], wires=1)
        qml.RZ(params[base + 3], wires=1)
        qml.CNOT(wires=[0, 1])
    return qml.probs(wires=range(N_QUBITS))

state_probs = final_circuit(result.x)
print("\nFinal state probabilities:")
for i, p in enumerate(state_probs):
    if p > 0.001:
        print(f"  |{i:02b}>  P = {p:.6f}")
print("\nCircuit:")
print(qml.draw(final_circuit)(result.x))